# Notebook: Automated Essay Grading System

## Overview

This notebook presents an Automated Essay Grading System designed to evaluate student answers against a set of key answers using a multi-faceted approach. The system leverages Natural Language Processing (NLP) techniques, including Sentence-BERT for semantic similarity, SpaCy for conceptual similarity and keyword extraction, and NLTK for text preprocessing.

### How it Works

1.  **Text Preprocessing**: Student answers are cleaned by converting to lowercase, tokenizing, removing stopwords, and stripping punctuation.
2.  **Semantic Similarity**: Uses a pre-trained Sentence-BERT model (`paraphrase-MiniLM-L6-v2`) to calculate the cosine similarity between the student's answer and multiple key answers. This captures the overall meaning similarity.
3.  **Conceptual Similarity**: Employs SpaCy's Word2Vec embeddings (from `en_core_web_lg`) to compute the conceptual similarity between the student's answer and key answers, focusing on deeper conceptual understanding.
4.  **Keyword Matching**: Extracts key words from the reference answers using SpaCy (`en_core_web_sm`) and then calculates a score based on how many of these keywords are present in the student's preprocessed answer.
5.  **Length Similarity**: Compares the length of the student's answer to the average length of the key answers, providing a heuristic score based on length parity.
6.  **Weighted Scoring**: The individual similarity scores (semantic, conceptual, keyword, and length) are combined using predefined weights to yield a `total_score`.
7.  **Final Grade Assignment**: A heuristic grading logic converts the `total_score` into a letter grade (e.g., "Outstanding", "Excellent", "Average", "Fail").

### Components

*   **`sentence_transformers`**: For generating semantic embeddings.
*   **`nltk`**: For tokenization and stopword removal.
*   **`spacy`**: For advanced NLP tasks like Word2Vec embeddings and part-of-speech tagging for keyword extraction.
*   **`pandas`**: For data handling, especially for loading key answers from an Excel file.
*   **`scikit-learn`**: For cosine similarity calculations.
*   **`torch`**: Underlying framework for `sentence_transformers`.

This system provides a robust framework for automatically assessing free-text responses, offering insights into various aspects of student comprehension.

In [ ]:
!pip install sentence_transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.3/163.3 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 49.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 60.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 71.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 10.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 MB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 12.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━

## Setup and Installations

This section handles the installation of necessary libraries and the downloading of NLP models and data required for the grading system.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
!pip install spacy

# Download the SpaCy Word2Vec model
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.7/587.7 MB 658.5 kB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## Library Imports and Model Loading

Here, all the required Python libraries are imported. This includes modules for web application development (Flask, though not fully utilized in this snippet), NLP tasks, data manipulation, and machine learning utilities.

In [ ]:
from flask import Flask, render_template, request, jsonify
from sentence_transformers import SentenceTransformer, InputExample, losses
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec
import pandas as pd
import string
import nltk
import torch
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from torch.utils.data import DataLoader
from torch import optim

## Model Initialization and Loss Function

This section initializes the `SentenceTransformer` model, which is central to calculating semantic similarity. It also includes a commented-out section for a custom `ContrastiveLoss` and a fine-tuning loop, demonstrating how the model could be further trained on custom data if needed. For this notebook, a pre-trained model is used.

In [ ]:
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from sentence_transformers import InputExample, SentenceTransformer
import torch.optim as optim

sbert_model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

#  Define optimizer and loss function
optimizer = optim.AdamW(sbert_model.parameters(), lr=2e-5)

'''# Define the modified ContrastiveLoss class
class ContrastiveLoss(nn.Module):
    def __init__(self, model):
        super(ContrastiveLoss, self).__init__()
        self.model = model

    def forward(self, sentence_features, labels):
        embeddings = sentence_features
        rep_anchor = torch.tensor(embeddings[0], requires_grad=True)  # Ensure gradient tracking
        rep_other = torch.tensor(embeddings[1], requires_grad=True)   # Ensure gradient tracking

        # Ensure correct tensor shapes
        rep_anchor = rep_anchor.unsqueeze(0)  # Add a batch dimension
        rep_other = rep_other.unsqueeze(0)    # Add a batch dimension

        # Compute similarity scores
        sim_scores = F.cosine_similarity(rep_anchor, rep_other, dim=1)

        # Convert labels to tensor
        labels = torch.tensor(labels, dtype=torch.float).unsqueeze(1)  # Add a dimension for broadcasting

        # Ensure correct tensor shapes for labels
        labels = labels.unsqueeze(1) if labels.dim() == 1 else labels

        # Calculate contrastive loss
        loss = F.mse_loss(sim_scores, labels)

        return loss

train_loss = ContrastiveLoss(sbert_model)

# Fine-tuning loop
num_epochs = 3
sbert_model.train()
for epoch in range(num_epochs):
    total_loss = 0

    for step, (texts, labels) in enumerate(dataloader):
        optimizer.zero_grad()
        # Encode the texts
        embeddings = sbert_model.encode(texts)
        # Calculate loss
        loss = train_loss(embeddings, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss}")

# Step 6: Save the fine-tuned model
sbert_model.save("/content/drive/MyDrive/fine_tuned_sbert_model")'''


'# Define the modified ContrastiveLoss class\nclass ContrastiveLoss(nn.Module):\n    def __init__(self, model):\n        super(ContrastiveLoss, self).__init__()\n        self.model = model\n\n    def forward(self, sentence_features, labels):\n        embeddings = sentence_features\n        rep_anchor = torch.tensor(embeddings[0], requires_grad=True)  # Ensure gradient tracking\n        rep_other = torch.tensor(embeddings[1], requires_grad=True)   # Ensure gradient tracking\n\n        # Ensure correct tensor shapes\n        rep_anchor = rep_anchor.unsqueeze(0)  # Add a batch dimension\n        rep_other = rep_other.unsqueeze(0)    # Add a batch dimension\n\n        # Compute similarity scores\n        sim_scores = F.cosine_similarity(rep_anchor, rep_other, dim=1)\n\n        # Convert labels to tensor\n        labels = torch.tensor(labels, dtype=torch.float).unsqueeze(1)  # Add a dimension for broadcasting\n\n        # Ensure correct tensor shapes for labels\n        labels = labels.unsq

## Utility Functions and Weights

This section defines global parameters and initializes NLTK's stopwords for text preprocessing. It also sets the weights for combining different similarity metrics into a final score, allowing for fine-tuning the influence of each component.

In [ ]:
# NLTK for tokenization and stop words
stop_words = set(stopwords.words('english'))

# Weights for each similarity metric during training
weight_length = 0.2
weight_semantic_similarity = 0.3
weight_conceptual_similarity = 0.2
weight_keyword_matching = 0.3

## Text Preprocessing Function

This function takes raw text as input and performs several cleaning steps crucial for accurate NLP analysis. It converts text to lowercase, tokenizes it, removes common English stopwords, and strips punctuation. This ensures that the downstream similarity calculations are based on meaningful content.

In [ ]:
def preprocess_text(text):
    # Convert to lowercase
    text_lower = text.lower()

    # Tokenization
    tokens = word_tokenize(text_lower)

    # Removing stop words
    tokens_filtered = [token for token in tokens if token.lower() not in stop_words]

    # Stripping of words
    table = str.maketrans('', '', string.punctuation)
    stripped = [token.translate(table) for token in tokens_filtered]
    stripped = [token for token in stripped if token.isalpha() or token.isdigit()]

    return ' '.join(stripped)

## Similarity Calculation Functions

This set of functions defines various ways to compare the student's answer with the key answers, covering different aspects of understanding: semantic meaning, conceptual depth, keyword presence, and answer length. Each function returns a score indicating the degree of similarity.

In [ ]:
def calculate_semantic_similarity(student_answer, key_answer):
    # Embedding for student answer
    answer_embedding = sbert_model.encode(student_answer, convert_to_tensor=True).reshape(1,-1)

    # Embeddings for key answers
    key_answer_embeddings = sbert_model.encode(key_answer, convert_to_tensor=True).reshape(1,-1)


    # Calculate cosine similarity
    semantic_similarity_score = cosine_similarity(answer_embedding, key_answer_embeddings).mean()

    # Heuristic grading logic for cosine similarity
    return (semantic_similarity_score)


In [ ]:
import spacy
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load SpaCy model
context_model = spacy.load("en_core_web_lg")

def calculate_conceptual_similarity(student_answer, key_ans_1, key_ans_2, key_ans_3):
    # Tokenize sentences using SpaCy
    student_tokens = context_model(student_answer)
    key1_tokens = context_model(key_ans_1)
    key2_tokens = context_model(key_ans_2)
    key3_tokens = context_model(key_ans_3)

    # Compute average vector representations for student answer
    student_vector = np.mean([token.vector for token in student_tokens if token.has_vector], axis=0)

    # Compute average vector representations for key answers
    key1_vector = np.mean([token.vector for token in key1_tokens if token.has_vector], axis=0)
    key2_vector = np.mean([token.vector for token in key2_tokens if token.has_vector], axis=0)
    key3_vector = np.mean([token.vector for token in key3_tokens if token.has_vector], axis=0)

    # Calculate cosine similarity for each key answer
    similarity_score_1 = cosine_similarity([student_vector], [key1_vector])[0][0]
    similarity_score_2 = cosine_similarity([student_vector], [key2_vector])[0][0]
    similarity_score_3 = cosine_similarity([student_vector], [key3_vector])[0][0]

    # Compute average similarity score
    average_similarity_score = (similarity_score_1 + similarity_score_2 + similarity_score_3) / 3

    return average_similarity_score



In [ ]:
import spacy

# Load the SpaCy model
nlp = spacy.load("en_core_web_sm")

def extract_key_words(text):
    # Tokenize and analyze the text using SpaCy
    doc = nlp(text)

    # Initialize an empty set to store key words
    key_words = set()

    # Extract relevant tokens based on part-of-speech tags and dependency relationships
    for token in doc:
        # Check if the token is a noun, proper noun, or adjective
        if token.pos_ in ["NOUN", "PROPN", "ADJ"] and not token.is_stop :
            # Add the token to the set of key words
            key_words.add(token.text.lower())  # Convert to lowercase for consistency

    return key_words

In [ ]:
def calculate_keyword_matching_score(key_ans_1,key_ans_2,key_ans_3,answer):

    topic_keywords=[]

    keys1=extract_key_words(key_ans_1)
    keys2=extract_key_words(key_ans_2)
    keys3=extract_key_words(key_ans_3)

    topic_keywords=keys1.intersection(keys2,keys3)

    matched_keywords = [keyword for keyword in topic_keywords if keyword in answer]

    # Calculate the matching score based on the number of matched keywords
    keyword_matching_score = len(matched_keywords) / len(topic_keywords)

    return keyword_matching_score

In [ ]:
def calculate_length_similarity(student_answer,avg_key_answer_length):
    # Normalize lengths
    if(len(student_answer)>avg_key_answer_length):
      length_difference=0.1
    else:
      max_length = max(len(student_answer), avg_key_answer_length)
      normalized_student_length = len(student_answer) / max_length
      normalized_key_length = avg_key_answer_length / max_length

      # Calculate absolute length difference
      length_difference = abs(normalized_student_length - normalized_key_length)

    # Sigmoid function - Nested function
    def score_func(x):
      return max(0, 1 - length_difference)


# Calculate length similarity score using sigmoid function
    length_similarity_score = score_func(length_difference)

    return length_similarity_score

## Overall Evaluation and Grading Logic

This `evaluate` function orchestrates the entire grading process. It takes a student's answer and multiple key answers, applies preprocessing, calculates all the individual similarity scores, and then combines them using the predefined weights to produce a single `total_score`. It also includes a threshold-based logic to handle cases where the answer might be too dissimilar or lack significant keywords.

In [ ]:
def evaluate(student_answer, key_ans_1, key_ans_2, key_ans_3):
    # Preprocess the student answer
    preprocessed_answer = preprocess_text(student_answer)

    # Calculate semantic similarity scores for each key answer
    semantic_similarity_score_1 = calculate_semantic_similarity(preprocessed_answer, key_ans_1)
    semantic_similarity_score_2 = calculate_semantic_similarity(preprocessed_answer, key_ans_2)
    semantic_similarity_score_3 = calculate_semantic_similarity(preprocessed_answer, key_ans_3)

    # Calculate the average semantic similarity score
    average_semantic_similarity_score = (semantic_similarity_score_1 + semantic_similarity_score_2 + semantic_similarity_score_3) / 3

    conceptual_similarity_score = calculate_conceptual_similarity(preprocessed_answer,key_ans_1,key_ans_2,key_ans_3)

    keyword_matching_score = calculate_keyword_matching_score(key_ans_1,key_ans_2,key_ans_3,preprocessed_answer)


    avg_key_answer_length=(len(key_ans_1)+len(key_ans_2)+len(key_ans_3))/3

    length_similarity_score=calculate_length_similarity(student_answer,avg_key_answer_length)

    if(average_semantic_similarity_score<0.2 or keyword_matching_score<0.1):
      total_score=0

    else:

      total_score = (
          weight_semantic_similarity * average_semantic_similarity_score +
          weight_conceptual_similarity * conceptual_similarity_score +
          weight_keyword_matching * keyword_matching_score +
          weight_length * length_similarity_score
      )

    # Grade based on total score
    return total_score

## Final Grade Assignment Function

This function translates the numerical `total_score` obtained from the `evaluate` function into a qualitative letter grade. It uses a set of heuristic thresholds to categorize scores into grades like "Outstanding", "Excellent", "Average", and "Fail", providing a human-readable assessment.

In [ ]:
def calculate_final_grade(total_score):
    # Heuristic grading logic for total score
    if total_score >=9:
        return "O : Outstanding"
    elif total_score >=8:
        return "A+ : Excellent"
    elif total_score >=7:
        return "A : Very Good"
    elif total_score >=6:
        return "B+ : Good"
    elif total_score >=5:
        return "B : Average"
    elif total_score >=3.5:
        return "C : Pass"
    else:
        return "F : Fail"

## Demonstration and Execution

This section showcases the automated grading system in action. It loads a list of key answers from an Excel file and a list of student answers. It then iterates through each student answer, evaluates it against the corresponding key answers using the `evaluate` function, accumulates the scores, and finally prints the aggregated total score and the overall final grade for the set of student answers.

In [ ]:
import math
key_answers_df=pd.read_excel('/content/drive/MyDrive/Key_Answers.xlsx')

student_answer_list = ["The primary key in a database is a unique identifier for each record in a table. It ensures that each row in the table is uniquely identifiable and can be used to establish relationships between tables. The primary key must be unique and not null for every record, and it is often chosen from one or more columns that have unique values.",
    "Database normalization is a process used to organize a relational database into tables and columns. It minimizes redundancy and dependency by dividing large tables into smaller ones and defining relationships between them. The objective is to isolate data so that additions, deletions, and modifications can be made in just one table and then propagated through the rest of the database via the defined relationships.",
    "Constraints in databases are rules that enforce data integrity and consistency. They define limits or conditions for the data that can be entered into a table, ensuring that only valid and acceptable data is stored. Common types of constraints include primary key constraints, foreign key constraints, unique constraints, check constraints, and not null constraints.",
    "Database generalization involves abstracting common properties of entities to create higher-level entities or concepts. It simplifies the database schema by identifying shared characteristics among different entities and creating a more generalized representation. This process improves data organization and facilitates data management, especially in hierarchical or inheritance-based data models where parent-child relationships are prevalent.",
    "Database specialization is the process of refining generalized entities into more specific entities or subclasses. It involves identifying unique characteristics or attributes of entities and creating specialized representations to accommodate them. Specialization is often used in hierarchical or inheritance-based data models to create subclass relationships between entities, allowing for more precise data modeling and organization.",
    "i dont know this answer",
    "SQL, or Structured Query Language, is a standard programming language used to communicate with and manipulate databases. It is used for tasks such as querying data, updating records, and defining database structures. SQL consists of various commands, including SELECT, INSERT, UPDATE, DELETE, and CREATE, which allow users to interact with relational databases efficiently. SQL is essential for database management and is widely used in applications ranging from simple websites to complex enterprise systems.",
    "The Entity-Relationship (ER) model is a graphical representation used to design and describe the structure of a database. It consists of entities, attributes, and relationships between entities. Entities represent real-world objects or concepts, attributes define the properties of entities, and relationships depict how entities are connected. The ER model provides a clear and concise way to visualize the data model and is widely used in database design and development.",
    "Indexing and hashing are techniques used to improve the efficiency of data retrieval in databases. Indexing involves creating data structures, such as B-trees or hash tables, that store references to records based on key values. This allows for faster data access by enabling the database to locate records quickly using index keys. Hashing, on the other hand, uses hash functions to map keys to storage locations, providing constant-time access to data. These techniques are essential for optimizing database performance and ensuring rapid query processing.",
    "I dont know this answer"]

key_ans_1_list = key_answers_df['Key_ans_1']
key_ans_2_list = key_answers_df['Key_ans_2']
key_ans_3_list = key_answers_df['Key_ans_3']
total_score=0

for student_answer, key_ans_1, key_ans_2, key_ans_3 in zip(student_answer_list, key_ans_1_list, key_ans_2_list, key_ans_3_list):
  total = evaluate(student_answer, key_ans_1, key_ans_2, key_ans_3)
  total_score+=total

print(round(total_score,2))
print(calculate_final_grade(total_score))

7.25
A : Very Good
